This script classifies LoD2 buildings as residential, non-residential, or auxiliary based on building function, footprint area, and whether a building is attached to neighboring buildings.

In [ ]:
"""
LoD2 Building Classification Script
===================================
Classifies buildings into three categories:
  1. residential       → function == "31001_1000"
  2. non_residential   → function != "31001_1000"
  3. auxiliary         → auxiliary buildings (overrides categories 1 and 2)

Auxiliary building criteria:
  - Standalone building (no neighboring buildings / no contact with other buildings)
    AND footprint area < 56.8 m²
  - OR attached to another building
    AND footprint area < 35 m²

Input:  LoD2_2025.gpkg
Output: LoD2_2025_classified.gpkg (new column: "building_class")
"""

import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union
import numpy as np

# ─────────────────────────────────────────────
# 1. Load data
# ─────────────────────────────────────────────
INPUT_PATH  = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025.gpkg"
OUTPUT_PATH = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_classified.gpkg"

print("Loading data...")
gdf = gpd.read_file(INPUT_PATH)
print(f"   {len(gdf)} buildings loaded | CRS: {gdf.crs}")

# ─────────────────────────────────────────────
# 2. Check projection (must use a projected CRS for area calculations)
# ─────────────────────────────────────────────
if gdf.crs is None:
    raise ValueError("No CRS defined. Please assign a CRS before proceeding.")

if gdf.crs.is_geographic:
    print("Geographic CRS detected → reprojecting to EPSG:25832 (UTM Zone 32N)")
    gdf = gdf.to_crs(epsg=25832)

# ─────────────────────────────────────────────
# 3. Calculate building footprint area (m²)
# ─────────────────────────────────────────────
gdf["area_m2"] = gdf.geometry.area
print(f"   Footprint area calculated | Min: {gdf['area_m2'].min():.1f} m²  Max: {gdf['area_m2'].max():.1f} m²")

# ─────────────────────────────────────────────
# 4. Spatial join: identify neighboring buildings (touching/intersecting)
#    → "attached" = geometry touches or intersects another building
# ─────────────────────────────────────────────
print("🔍 Identifying neighboring buildings (spatial join)...")

# Apply a small buffer (5 cm) to robustly detect touching buildings
gdf_buf = gdf.copy()
gdf_buf["geometry"] = gdf_buf.geometry.buffer(0.05)

joined = gpd.sjoin(
    gdf_buf[["geometry"]].reset_index().rename(columns={"index": "idx_left"}),
    gdf_buf[["geometry"]].reset_index().rename(columns={"index": "idx_right"}),
    how="left",
    predicate="intersects"
)

# Remove self-matches
joined = joined[joined["idx_left"] != joined["idx_right"]]

# Buildings with at least one neighboring building
has_neighbor = set(joined["idx_left"].unique())

gdf["has_neighbor"] = gdf.index.isin(has_neighbor)
print(f"   {gdf['has_neighbor'].sum()} buildings have at least one neighbor")
print(f"   {(~gdf['has_neighbor']).sum()} buildings are standalone")

# ─────────────────────────────────────────────
# 5. Building classification
# ─────────────────────────────────────────────
print("🏗️ Classifying buildings...")

RESIDENTIAL_CODE  = "31001_1000"
THRESH_STANDALONE = 56.8   # m² – standalone buildings
THRESH_ATTACHED   = 35.0   # m² – attached buildings

def classify(row):
    area         = row["area_m2"]
    has_neighbor = row["has_neighbor"]

    # Auxiliary classification takes precedence over function-based classification
    if not has_neighbor and area < THRESH_STANDALONE:
        return "auxiliary"
    if has_neighbor and area < THRESH_ATTACHED:
        return "auxiliary"

    # Residential vs. non-residential based on the function attribute
    func = str(row.get("function", "")).strip()
    if func == RESIDENTIAL_CODE:
        return "residential"
    else:
        return "non_residential"

gdf["building_class"] = gdf.apply(classify, axis=1)

# ─────────────────────────────────────────────
# 6. Summary statistics
# ─────────────────────────────────────────────
print("\n📊 Classification summary:")
summary = gdf["building_class"].value_counts()
for cls, count in summary.items():
    pct = count / len(gdf) * 100
    print(f"   {cls:<20} {count:>6} buildings ({pct:.1f}%)")

# ─────────────────────────────────────────────
# 7. Save results
# ─────────────────────────────────────────────
# Remove helper columns here if desired
gdf_out = gdf

print(f"\nSaving to: {OUTPUT_PATH}")
gdf_out.to_file(OUTPUT_PATH, driver="GPKG")
print("Done!")